<a href="https://colab.research.google.com/github/AcSsalazar/the-color-of-emotions/blob/main/Notebooks-EN/4.4-training-building-a-cnn-mfccs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Bidirectional Gated Recurrent Unit (BiGRU) Implementation for MFCC-Based Tensors.

In this notebook we will use a *Bidirectional Gated Recurrent Unit* (BiGRU), an advanced variant of Recurrent Neural Networks (RNNs) that improves sequence processing by analyzing data in both directions: from start to end (forward) and from end to start (backward). This architecture combines two independent GRU layers to capture both past and future context for each element in a sequence. We use it to take advantage of the channels that preserve temporal dimension in our tensors, namely the MFCC delta and second-order delta ($\Delta^2$) channels.

It is worth noting that for the first blocks of the neural network architecture, we use standard convolutional units, so our architecture has the following base structure:

## Monitoring with Weights & Biases

To optimize the performance of our CBGRU (CNN + BiGRU) architecture, we integrate Weights & Biases (W&B) as the central experimentation platform. In complex hybrid models, it is critical to understand how the convolutional and recurrent layers interact.

**Tracking**: Real-time logging of metrics (loss, accuracy), hyperparameters, and hardware usage (GPU/CPU).

**Visualization**: Interactive dashboards to compare runs and analyze model performance.

**Artifacts**: Version control for datasets and models, ensuring traceability and reproducibility.

**Sweeps**: Automation to search for the best hyperparameters to optimize the model.

The goal is:

* Log each run (hyperparameters + metrics + artifacts)

* Select the best checkpoint using validation only.

* Evaluate only once at the end to avoid test data leakage.

* Verify the evolution of models and their different architectures.

* Obtain a final model for inference and deployment after verifying performance from the experimentation.

### 1.0 Imports and base configuration


In [ ]:
# Imports
#------------------------------------------------------------------------------------------
import os
import copy
import numpy as np
import torch
import xgboost as xgb
import random as random
import torch.nn as nn
import matplotlib.pyplot as plt
import pandas as pd
import time
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torchaudio import transforms as T
from torch.optim.lr_scheduler import ReduceLROnPlateau
from sklearn.model_selection import GridSearchCV, PredefinedSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                             ConfusionMatrixDisplay,f1_score, accuracy_score)
from google.colab import drive
from tqdm import tqdm
from collections import Counter

In [ ]:
# ─── Weights & Biases installation ────────────────────────────────
# Uncomment the following lines if wandb is not installed in your environment:
!pip install wandb -q

# To authenticate (only the first time per Colab session):
import wandb; wandb.login()

# USE_WANDB is defined in the configuration cell below.
# Leave it False to run without remote tracking.

In [ ]:
# Semilla y runtime
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

### 1.2 Load tensors from Drive (train/val/test)

The `.pt` files generated in notebook 3.2 contain a dictionary with `x`, `y`, metadata, and the class mapping. Adjust `BASE_DIR_TENSOR` according to your local/Drive path.

In [ ]:
drive.mount('/content/drive')
! cp -r /content/drive/MyDrive/split_pytorch_tensors_mfcc /content/split_pytorch_tensors

In [ ]:
BASE_DIR_TENSOR = '/content/split_pytorch_tensors'
BATCH_SIZE = 32

SPLIT_FILES = {
    'train': 'train_tensors.pt',
    'val': 'val_tensors.pt',
    'test': 'test_tensors.pt',
}

def load_pack(split_name: str):
    path = os.path.join(BASE_DIR_TENSOR, SPLIT_FILES[split_name])
    if not os.path.exists(path):
        raise FileNotFoundError(f'File does not exist: {path}')
    return torch.load(path, map_location='cpu', weights_only=False)

# Load the packs into variables; they are not pure tensors yet
train_pack = load_pack('train')
val_pack = load_pack('val')
test_pack = load_pack('test')

# Verify the classes in the dictionary inside the pack
class_to_idx = train_pack['class_to_idx']
idx_to_class = {v: k for k, v in class_to_idx.items()}
class_names = [idx_to_class[i] for i in sorted(idx_to_class.keys())]
print('Clases:', class_names)
print('Shape train:', tuple(train_pack['x'].shape))

### 1.3 Normalization and augmentation strategy

**Normalization:**  
The tensors come from notebook `3.2` and were already normalized with **per-sample and per-channel z-score** (`zscore_per_channel`). No second dataset-level normalization is applied to avoid double normalization.

**Online augmentation (disabled by default):**  
Notebook `3.2` already added extra samples of `surprised` with **noise** and **time shift** during export (offline augmentation). Therefore `augment=False` is the default value in `build_dataloaders`: offline augmentation already provides robustness without the risk of excessively distorting inputs.  
If additional online augmentation is desired, `augment=True` can be enabled; in that case only **SpecAugment** (FrequencyMasking + TimeMasking) is applied, plus a small optional Gaussian noise.

**Early stopping and scheduler:**  
**macro-F1** is used as the criterion instead of `val_loss`, since it is more informative for imbalanced datasets and better reflects the goal of balanced classification across classes.

In [ ]:
class TensorPackDataset(Dataset):
  def __init__(self, pack, augment=False):
      self.x = pack['x'].float() # [N, 3, n_mfcc, targetframes]
      self.y = pack['y'].long()  # len(N)
      self.augment = augment

      # Define the SpecAugment transformations
      # Adjust n_freq_masks and n_time_masks based on size (13x51)
      if augment:
          self.spec_aug = nn.Sequential(
              T.FrequencyMasking(freq_mask_param=3), # Enmascara hasta 4 bins de mel
              T.TimeMasking(time_mask_param=1)       # Enmascara hasta 2 frames de tiempo
          )

  def __len__(self):
      return self.y.shape[0]

  def __getitem__(self, idx):
      x = self.x[idx]
      y = self.y[idx]

      if self.augment:
          # SpecAugment espera [batch, channel, freq, time] o [channel, freq, time]
          # Apply the same mask to all 3 channels (MFCC, Delta, Delta2)
          x = self.spec_aug(x)
          '''
          # Optional: small Gaussian noise
          if random.random() < 0.5:
              x = x + 0.01 * torch.randn_like(x)'''

      return x, y

### 1.4 Early stop configuration

In [ ]:
class EarlyStopping:
    """Configurable early stopping to minimize (val_loss) or maximize (val_f1)."""
    def __init__(self, patience=5, min_delta=0.0, mode="min"):
        self.patience = patience
        self.min_delta = min_delta
        self.mode = mode  # "min" for loss, "max" for F1
        self.best = None
        self.counter = 0

    def step(self, metric):
        """Returns True when training should stop."""
        if self.best is None:
            self.best = metric
            return False
        improved = (metric > self.best + self.min_delta) if self.mode == "max" \
                   else (metric < self.best - self.min_delta)
        if improved:
            self.best = metric
            self.counter = 0
            return False
        self.counter += 1
        print(f"Early Stopping: {self.counter}/{self.patience} (best={self.best:.4f})")
        return self.counter >= self.patience


### 1.5 Dataloaders

In [ ]:
# We only augment the TRAIN set. Val and Test must remain clean.
def build_dataloaders(batch_size=BATCH_SIZE):
    pin = torch.cuda.is_available()
    # Default value: augment=False
    train_ds = TensorPackDataset(train_pack, augment=False)
    val_ds = TensorPackDataset(val_pack, augment=False)
    test_ds = TensorPackDataset(test_pack, augment=False)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=pin)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=pin)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=pin)

    return train_loader, val_loader, test_loader

train_loader, val_loader, test_loader = build_dataloaders()


### 1.6 CGRU definition


In [ ]:
class EmotionCGRU(nn.Module):
    def __init__(self, num_classes, classifier_dropout=0.5):
        super().__init__()

        # Expected input: (Batch, 3, 13, 51)
        self.cnn = nn.Sequential(
            # Bloque Convolucional 1
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            # Reduce frequency smoothly, keep time intact
            nn.MaxPool2d(kernel_size=(2, 1)), # Frec: 13 -> 6, Tiempo: 51 -> 51

            # Bloque Convolucional 2
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=(2, 1)), # Frec: 6 -> 3, Tiempo: 51 -> 51
            nn.Dropout2d(0.1),

            # Bloque Convolucional 3
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Dropout2d(0.25)
            # CNN output: (Batch, 64, 3, 51)
        )

        # 64 canales * 3 bins de frecuencia restantes = 192 features por frame
        self.gru_input_size = 64 * 3

        self.bigru = nn.GRU(
            input_size=self.gru_input_size,
            hidden_size=64,
            num_layers=2,
            bidirectional=True,
            batch_first=True,
            dropout=0.25
        )

        self.classifier = nn.Sequential(
            # 64 * 2 porque la GRU es bidireccional
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(classifier_dropout),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        # 1. CNN feature extraction
        x = self.cnn(x)

        batch_size, channels, freq, time = x.size()

        # 2. Format for GRU: (Batch, Time, Features)
        # Permute to (Batch, Time, Channels, Frequency) -> (Batch, 51, 64, 3)
        x = x.permute(0, 3, 1, 2).contiguous()
        # Flatten to (Batch, 51, 192)
        x = x.view(batch_size, time, channels * freq)

        # 3. Sequential modeling
        gru_out, _ = self.bigru(x)

        # 4. Temporal pooling (average) and classification
        x = gru_out.mean(dim=1)
        return self.classifier(x)

## Tracking with Weights & Biases (W&B)

In Colab/Jupyter notebooks, re-running cells often overwrites variables and results, which makes it easy to lose:

* Which hyperparameters produced the best model and the full training curves (loss/F1 vs. epoch).

* The best checkpoint for that run, the confusion matrix, and the classification report associated with that checkpoint.

***W&B*** solves this by storing runs in an external experiment log, where each run includes:

1. Configuration aspects `(wandb.config)` such as `dropout`, `seeds`, `weight decay`, `learning rate`, etc.

2. The time series of epoch metrics `train val loss`, `macro-F1`, `LR`.

3. Artifacts such as model checkpoints and reports.

### 2.0 W&B configuration

The baseline is built from previous experimentation where greater stability was achieved with `BATCH_SIZE=32`, and—as mentioned earlier—without *online data augmentation*, although these features are preserved for research purposes for future users.

In [ ]:
# --- W&B configuration and experiment matrix -----------------------------

# Set USE_WANDB = True to log experiments in Weights & Biases.
# Requires wandb installed (!pip install wandb -q) and authenticated with an API key.
USE_WANDB      = True                # ← set to True to enable tracking
WANDB_PROJECT  = "tcoe-experiments-no-aug-mfccs"
WANDB_GROUP    = "cnn-cgru-batch32-2.2"  # groups the four experiments in the same view (very useful)

# --- Hyperparameters fixed across the four experiments -------------------

# Confirmed baseline: BATCH_SIZE=32, augment=False (no online SpecAugment),

EPOCHS              = 50
LABEL_SMOOTHING     = 0.05
MAX_GRAD_NORM       = 1.0
EARLY_STOP_PATIENCE = 8    # higher patience so LR drops take effect; do not use a value lower than 4

# Local directory to save the best checkpoints per run
CHECKPOINT_DIR = '/content/checkpoints_cnn'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# --- Experiment matrix (4 runs) ------------------------------------
# Fixed scheduler: ReduceLROnPlateau(mode='max', factor=0.5, patience=2) on validation macro-F1.
# Gradient clipping: max_norm=1.0. Label smoothing: 0.05.

RUN_MATRIX = [
    # Run 1 — strong baseline: conservative LR, standard dropout, class weights
    {
        "run_name": "run1_lr3e4_wd1e3_do05_cw",
        "lr": 3e-4, "weight_decay": 1e-3,
        "classifier_dropout": 0.5,
        "use_class_weights": True,
    },
    # Run 2 — higher LR: explore faster convergence
    {
        "run_name": "run2_lr5e4_wd1e3_do05_cw",
        "lr": 5e-4, "weight_decay": 1e-3,
        "classifier_dropout": 0.6,
        "use_class_weights": False,
    },
    # Run 3 — reduced dropout: less regularization in the classifier
    {
        "run_name": "run3_lr3e4_wd1e3_do03_cw",
        "lr": 3e-4, "weight_decay": 1e-3,
        "classifier_dropout": 0.3,
        "use_class_weights": True,
    },
    # Run 4 — diagnostic: no class weights (measures the impact of balancing)
    {
        "run_name": "run4_lr3e4_wd1e3_do05_nocw",
        "lr": 3e-4, "weight_decay": 1e-3,
        "classifier_dropout": 0.5,
        "use_class_weights": False,
    },
]


### 4. Training and validation


In [ ]:
# TRAIN
#-------------------------------------------------------------------------------
# Compute class weights: total / (n_classes * counts)
# This gives more weight to classes with fewer samples.
''''surprised' has extra samples from offline export, so its resulting weight
    will be lower than other under-represented classes.'''

y_train_cpu = train_pack['y'].numpy()
label_counts = Counter(y_train_cpu)
total_samples = len(y_train_cpu)
num_classes = len(class_names)

class_weights = []
for i in range(num_classes):
    count = label_counts.get(i, 1)
    weight = total_samples / (num_classes * count)
    class_weights.append(weight)

class_weights_tensor = torch.FloatTensor(class_weights).to(device)
print(f"Weights calculated for CrossEntropyLoss: {class_weights}")


def train_one_epoch(model, loader, criterion, optimizer, device, epoch=None, max_grad_norm=1.0):
    model.train()
    running_loss = 0.0
    running_correct = 0

    pbar = tqdm(loader, desc=f"Epoch {epoch:02d} [train]", leave=False)
    for inputs, labels in pbar:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        # Gradient clipping to stabilize GRU training
        if max_grad_norm is not None:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        running_correct += (outputs.argmax(1) == labels).sum().item()
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = running_correct / len(loader.dataset)
    return epoch_loss, epoch_acc


# VAL
#-------------------------------------------------------------------------------
@torch.no_grad()
def evaluate(model, loader, criterion, device, epoch=None):
    model.eval()
    running_loss = 0.0
    running_correct = 0
    all_preds, all_targets = [], []

    pbar = tqdm(loader, desc=f"Epoch {epoch:02d} [val]", leave=False)
    for inputs, labels in pbar:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        running_loss += loss.item() * inputs.size(0)
        running_correct += (outputs.argmax(1) == labels).sum().item()
        all_preds.extend(outputs.argmax(1).cpu().numpy())
        all_targets.extend(labels.cpu().numpy())
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = running_correct / len(loader.dataset)
    # Macro-F1: key metric for imbalanced datasets with an even classification objective
    epoch_f1 = f1_score(all_targets, all_preds, average='macro', zero_division=0)
    return epoch_loss, epoch_acc, epoch_f1


# TEST / PREDICTIONS
#-------------------------------------------------------------------------------
@torch.no_grad()
def get_predictions(model, loader, device):
    model.eval()
    all_preds, all_targets = [], []
    for inputs, labels in loader:
        outputs = model(inputs.to(device))
        all_preds.append(outputs.argmax(1).cpu().numpy())
        all_targets.append(labels.numpy())
    return np.concatenate(all_targets), np.concatenate(all_preds)


In [ ]:
# ─── Multi-experiment training with W&B ──────────────────────────────────
all_run_results = []

for run_cfg in RUN_MATRIX:
    run_name           = run_cfg["run_name"]
    lr                 = run_cfg["lr"]
    weight_decay       = run_cfg["weight_decay"]
    classifier_dropout = run_cfg["classifier_dropout"]
    use_class_weights  = run_cfg["use_class_weights"]

    print(f"\n{'='*70}")
    print(f"  INICIANDO: {run_name}")
    print(f"  lr={lr}  wd={weight_decay}  dropout={classifier_dropout}  class_weights={use_class_weights}")
    print(f"{'='*70}\n")

    # Reproducibility per run
    random.seed(42)
    np.random.seed(42)
    torch.manual_seed(42)
    torch.cuda.manual_seed_all(42)

    # Fresh dataloaders for each run
    train_loader, val_loader, test_loader = build_dataloaders()

    # Model with classifier dropout depending on the run
    model = EmotionCGRU(num_classes=len(class_names), classifier_dropout=classifier_dropout).to(device)

    # Loss function
    criterion_weights = class_weights_tensor if use_class_weights else None
    criterion = nn.CrossEntropyLoss(weight=criterion_weights, label_smoothing=LABEL_SMOOTHING)

    # Optimizer and scheduler
    optimizer    = AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler    = ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)
    early_stopper = EarlyStopping(patience=EARLY_STOP_PATIENCE, min_delta=1e-4, mode='max')

    # Initialize W&B
    if USE_WANDB:
        import wandb
        wandb.init(
            project=WANDB_PROJECT,
            group=WANDB_GROUP,
            name=run_name,
            config={
                "lr":                 lr,
                "weight_decay":       weight_decay,
                "classifier_dropout": classifier_dropout,
                "use_class_weights":  use_class_weights,
                "label_smoothing":    LABEL_SMOOTHING,
                "batch_size":         BATCH_SIZE,
                "epochs":             EPOCHS,
                "early_stop_patience": EARLY_STOP_PATIENCE,
                "max_grad_norm":      MAX_GRAD_NORM,
                "augment_online":     True,
                "model":              "EmotionCGRU",
            },
        )

    best_val_f1 = 0.0
    best_epoch  = 0
    best_state  = copy.deepcopy(model.state_dict())

    for epoch in range(1, EPOCHS + 1):
        start = time.time()

        train_loss, train_acc = train_one_epoch(
            model, train_loader, criterion, optimizer, device,
            epoch=epoch, max_grad_norm=MAX_GRAD_NORM,
        )
        val_loss, val_acc, val_f1 = evaluate(
            model, val_loader, criterion, device, epoch=epoch,
        )

        scheduler.step(val_f1)
        current_lr = optimizer.param_groups[0]['lr']
        elapsed = time.time() - start

        print(
            f"[{time.strftime('%H:%M:%S')}] Epoca {epoch:02d}/{EPOCHS:02d} | "
            f"Train loss {train_loss:.4f} acc {train_acc:.4f} | "
            f"Val loss {val_loss:.4f} acc {val_acc:.4f} f1 {val_f1:.4f} | "
            f"lr {current_lr:.1e} | Time {elapsed:.1f}s"
        )

        if USE_WANDB:
            wandb.log({
                "train_loss":   train_loss,
                "train_acc":    train_acc,
                "val_loss":     val_loss,
                "val_acc":      val_acc,
                "val_macro_f1": val_f1,
                "lr":           current_lr,
            }, step=epoch)

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_epoch  = epoch
            best_state  = copy.deepcopy(model.state_dict())
            ckpt_path   = os.path.join(CHECKPOINT_DIR, f"{run_name}_best.pth")
            torch.save(best_state, ckpt_path)

        if early_stopper.step(val_f1):
            print(f"Early stopping en epoch {epoch} (best val_f1={best_val_f1:.4f})")
            break

    # ── Test evaluation (one time only, with the best model by val_f1) ──────
    model.load_state_dict(best_state)
    y_true, y_pred = get_predictions(model, test_loader, device)
    test_f1  = f1_score(y_true, y_pred, average='macro', zero_division=0)
    test_acc = accuracy_score(y_true, y_pred)

    print(f"\n--- RESULTADO TEST [{run_name}] ---")
    print(f"  best_epoch={best_epoch}  best_val_f1={best_val_f1:.4f}")
    print(f"  test_macro_f1={test_f1:.4f}  test_acc={test_acc:.4f}")
    print(classification_report(y_true, y_pred, target_names=class_names))

    # Confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    fig_cm, ax_cm = plt.subplots(figsize=(8, 6))
    ConfusionMatrixDisplay(cm, display_labels=class_names).plot(
        ax=ax_cm, cmap='Blues', xticks_rotation=45, values_format='d')
    ax_cm.set_title(f"Confusion Matrix — {run_name}")
    plt.tight_layout()
    plt.show()

    if USE_WANDB:
        # Save and log the checkpoint as an artifact
        artifact = wandb.Artifact(name=f"model_{run_name}", type="model")
        artifact.add_file(ckpt_path)
        wandb.log_artifact(artifact)

        # Test metrics and artifacts
        report_dict = classification_report(
            y_true, y_pred, target_names=class_names, output_dict=True)
        wandb.log({
            "test_macro_f1":       test_f1,
            "test_acc":            test_acc,
            "confusion_matrix":    wandb.Image(fig_cm),
            "classification_report": wandb.Table(
                columns=["class", "precision", "recall", "f1-score", "support"],
                data=[
                    [c,
                     report_dict[c]["precision"],
                     report_dict[c]["recall"],
                     report_dict[c]["f1-score"],
                     int(report_dict[c]["support"])]
                    for c in class_names
                ],
            ),
        })
        wandb.finish()

    all_run_results.append({
        "run_name":      run_name,
        "best_val_f1":   round(best_val_f1, 4),
        "best_epoch":    best_epoch,
        "test_macro_f1": round(test_f1, 4),
        "test_acc":      round(test_acc, 4),
    })
    plt.close('all')


### 5. Experiment summary and final evaluation


In [ ]:
# Summary table of the experiments -----------------------------------------------
summary_df = pd.DataFrame(all_run_results)
summary_df = summary_df.sort_values("best_val_f1", ascending=False).reset_index(drop=True)

print("\n" + "="*70)
print("  RESUMEN FINAL DE EXPERIMENTOS")
print("="*70)
print(summary_df.to_string(index=False))
display(summary_df)
